# Processo completo VRUM: preparação, modelagem e política

Este notebook parte das bases originais em `docs/` e percorre o fluxo completo: auditoria do target, preparação da base, features históricas sem vazamento, split temporal de 30 dias, XGBoost e política `APROVAR | INVESTIGAR | BLOQUEAR`.

A pergunta central é operacional: o score consegue separar risco futuro com estabilidade suficiente para apoiar uma decisão?

## As bases originais foram lidas corretamente?

Validamos separador, dimensões, tipos e amostras antes de unir qualquer tabela.

In [1]:
from datetime import timedelta
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from sklearn.metrics import average_precision_score, precision_recall_curve, precision_score, recall_score, roc_auc_score, roc_curve
from xgboost import XGBClassifier

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "docs" / "eventos_target_chassi_90d.csv").exists():
    REPO_ROOT = REPO_ROOT.parent
DOCS = REPO_ROOT / "docs"
OUTPUT = REPO_ROOT / "output"
COR_PRIMARIA, COR_SECUNDARIA, COR_DESTAQUE = "#0072B2", "#94A3B8", "#E69F00"
plt.rcParams.update({"figure.figsize": (10, 6), "figure.dpi": 110, "font.size": 11, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25})

propostas = pl.concat([pl.read_csv(DOCS / f"financiamentos_chassi_2026_{mes:02d}.csv", separator=";", null_values=[""], try_parse_dates=False, infer_schema_length=10_000) for mes in [1, 2, 3]])
eventos = pl.read_csv(DOCS / "eventos_target_chassi_90d.csv", separator=";", null_values=[""], try_parse_dates=False, infer_schema_length=10_000)
cadastro = pl.read_csv(DOCS / "cadastro_chassi_mock.csv", separator=";", null_values=[""], try_parse_dates=False, infer_schema_length=10_000)
propostas = propostas.with_columns(pl.col("data_hora_proposta").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False))
print("Arquivos carregados:", propostas.shape, eventos.shape, cadastro.shape)

Arquivos carregados: (2129214, 11) (2129213, 4) (600000, 6)


In [2]:
print("Schema propostas:", propostas.schema)
print("Schema eventos:", eventos.schema)
propostas.head(3)

Schema propostas: Schema({'id_proposta': String, 'data_hora_proposta': Datetime(time_unit='us', time_zone=None), 'chassi_id_sintetico': String, 'if_id_sintetico': String, 'cpf_cnpj_proponente_sintetico': String, 'tipo_proponente': String, 'valor_financiado': Float64, 'valor_entrada': Float64, 'prazo_meses': Int64, 'canal': String, 'uf_proposta': String})
Schema eventos: Schema({'id_proposta': String, 'evento_risco_chassi_90d': Int64, 'tipo_evento': String, 'dias_ate_evento': Int64})


id_proposta,data_hora_proposta,chassi_id_sintetico,if_id_sintetico,cpf_cnpj_proponente_sintetico,tipo_proponente,valor_financiado,valor_entrada,prazo_meses,canal,uf_proposta
str,datetime[μs],str,str,str,str,f64,f64,i64,str,str
"""PROP_CHASSI_0000000001""",2026-01-10 05:36:55,"""CHASSI_SYN_000035285""","""IF_SYN_09""","""DOC_PROP_SYN_000376285""","""PF""",125170.86,48209.12,36,"""DIGITAL""","""RJ"""
"""PROP_CHASSI_0000000002""",2026-01-18 14:21:04,"""CHASSI_SYN_000375721""","""IF_SYN_18""","""DOC_PROP_SYN_000235987""","""PF""",34455.69,14855.02,24,"""DIGITAL""","""MG"""
"""PROP_CHASSI_0000000003""",2026-01-29 22:43:19,"""CHASSI_SYN_000416248""","""IF_SYN_15""","""DOC_PROP_SYN_000297726""","""PF""",72615.57,16251.28,60,"""CONCESSIONARIA""","""SP"""


**Resultado:** as três fontes de financiamento têm a mesma estrutura e cobrem janeiro a março de 2026. A tabela de eventos tem uma linha por proposta esperada, mas a auditoria seguinte verifica se essa cobertura é completa.

## O target está completo, válido e semanticamente claro?

Auditamos duplicidade, órfãos, valores do target, janela temporal e propostas sem registro de evento.

In [3]:
eventos_por_id = eventos.group_by("id_proposta").agg(pl.len().alias("n"))
ids_propostas = propostas.select("id_proposta").unique()
auditoria_target = pl.DataFrame({
    "propostas": [propostas.height],
    "ids_eventos": [eventos.select("id_proposta").n_unique()],
    "eventos_duplicados": [eventos_por_id.filter(pl.col("n") > 1).height],
    "eventos_sem_proposta": [eventos.join(ids_propostas, on="id_proposta", how="anti").height],
    "propostas_sem_evento": [propostas.join(eventos.select("id_proposta").unique(), on="id_proposta", how="anti").height],
})
print(auditoria_target)
eventos.group_by(["evento_risco_chassi_90d", "tipo_evento"]).agg(pl.len().alias("n")).sort("n", descending=True)

shape: (1, 5)
┌───────────┬─────────────┬────────────────────┬──────────────────────┬──────────────────────┐
│ propostas ┆ ids_eventos ┆ eventos_duplicados ┆ eventos_sem_proposta ┆ propostas_sem_evento │
│ ---       ┆ ---         ┆ ---                ┆ ---                  ┆ ---                  │
│ i64       ┆ i64         ┆ i64                ┆ i64                  ┆ i64                  │
╞═══════════╪═════════════╪════════════════════╪══════════════════════╪══════════════════════╡
│ 2129214   ┆ 2129213     ┆ 0                  ┆ 0                    ┆ 1                    │
└───────────┴─────────────┴────────────────────┴──────────────────────┴──────────────────────┘


evento_risco_chassi_90d,tipo_evento,n
i64,str,u32
0,"""SEM_EVENTO""",2098019
1,"""FRAUDE_CONFIRMADA""",10408
1,"""BUSCA_APREENSAO_INFRUTIFERA""",10403
1,"""NEVER_PAY""",10383


In [4]:
rotulos = propostas.join(eventos, on="id_proposta", how="left")
risco = rotulos.filter(pl.col("evento_risco_chassi_90d") == 1).select(["id_proposta", "data_hora_proposta", "dias_ate_evento", "tipo_evento"])
risco = risco.with_columns(((pl.col("data_hora_proposta") + pl.duration(days=pl.col("dias_ate_evento"))) - pl.col("data_hora_proposta")).dt.total_seconds().truediv(86_400).alias("dias_calculados"))
risco.select([pl.len().alias("eventos_risco"), pl.col("dias_calculados").min().alias("min_dias"), pl.col("dias_calculados").max().alias("max_dias"), pl.col("dias_calculados").is_null().sum().alias("datas_invalidas")])

eventos_risco,min_dias,max_dias,datas_invalidas
u32,f64,f64,u32
31194,1.0,90.0,0


**Resultado:** target tem `SEM_EVENTO` explícito para quase toda a base, eventos de risco entre 1 e 90 dias e uma proposta sem qualquer linha de evento. Essa proposta não deve ser forçada para classe legítima: será marcada como `target_nao_observado` e excluída do treino.

## Qual base de modelagem preserva o target observado e o contexto do veículo?

Ligamos o cadastro estático ao histórico e mantemos target, tipo de evento e data do risco fora das features.

In [5]:
base = (rotulos.join(cadastro.select(["chassi_id_sintetico", "marca", "ano_modelo", "uf_registro", "valor_fipe_referencia"]), on="chassi_id_sintetico", how="left").with_columns([
    pl.col("evento_risco_chassi_90d").is_not_null().cast(pl.Int8).alias("target_observado"),
    pl.col("evento_risco_chassi_90d").alias("target_risco_90d"),
]).sort(["chassi_id_sintetico", "data_hora_proposta", "id_proposta"]))
base.select([pl.len().alias("linhas"), pl.col("target_observado").sum().alias("rotulos_observados"), pl.col("target_risco_90d").sum().alias("positivos"), pl.col("marca").is_null().sum().alias("cadastros_sem_match")])

linhas,rotulos_observados,positivos,cadastros_sem_match
u32,i64,i64,u32
2129214,2129213,31194,0


A proposta sem evento permanece no histórico para não apagar informação temporal, mas não participa da avaliação supervisionada. O cadastro entra como contexto estático; não representa informação futura.

## As features históricas respeitam o passado da proposta?

Calculamos intervalos, mediana acumulada e janelas exclusivas. Todas as operações usam somente propostas anteriores do mesmo chassi.

In [6]:
def adicionar_janela(df, dias):
    nome = f"transferencias_ultimos_{dias}d"
    janela = (df.select(["chassi_id_sintetico", "data_hora_proposta"]).rolling(index_column="data_hora_proposta", period=f"{dias}d", group_by="chassi_id_sintetico", closed="left").agg(pl.col("data_hora_proposta").count().alias(nome)).unique(["chassi_id_sintetico", "data_hora_proposta"]))
    return df.join(janela, on=["chassi_id_sintetico", "data_hora_proposta"])

In [7]:
def construir_features(df):
    df = df.sort(["chassi_id_sintetico", "data_hora_proposta", "id_proposta"])
    df = df.with_columns([
        ((pl.col("data_hora_proposta") - pl.col("data_hora_proposta").shift(1).over("chassi_id_sintetico")).dt.total_seconds() / 86_400).alias("tempo_posse_dias"),
        pl.col("id_proposta").cum_count().over("chassi_id_sintetico").sub(1).alias("qtd_propostas_historicas"),
        pl.col("tipo_proponente").shift(1).over("chassi_id_sintetico").alias("tipo_proponente_anterior"),
    ]).with_columns([
        (pl.col("tipo_proponente").is_not_null() & pl.col("tipo_proponente_anterior").is_not_null() & (pl.col("tipo_proponente") != pl.col("tipo_proponente_anterior"))).cast(pl.Int8).alias("flag_alternancia_pf_pj"),
        (2026 - pl.col("ano_modelo")).alias("idade_veiculo_anos"),
        (pl.col("valor_financiado") / pl.col("valor_fipe_referencia")).alias("ltv_fipe"),
    ])
    return df

In [8]:
features = construir_features(base)
features = features.with_columns(pl.col("tempo_posse_dias").shift(1).cumulative_eval(pl.element().drop_nulls().quantile(0.5, interpolation="linear"), min_samples=1).over("chassi_id_sintetico").fill_null(0).alias("tempo_posse_mediano_acumulado"))
for dias in [7, 15, 30, 45, 90]:
    features = adicionar_janela(features, dias)
features = features.with_columns([
    (pl.col("transferencias_ultimos_45d") / (pl.col("tempo_posse_mediano_acumulado") + 1)).alias("indice_rotatividade_vrum"),
    pl.col("tempo_posse_dias").alias("dias_desde_ultima_proposta"),
]).drop(["tempo_posse_dias", "tipo_proponente_anterior"])
features.select(["chassi_id_sintetico", "data_hora_proposta", "qtd_propostas_historicas", "tempo_posse_mediano_acumulado", "transferencias_ultimos_45d", "indice_rotatividade_vrum"]).head(5)

chassi_id_sintetico,data_hora_proposta,qtd_propostas_historicas,tempo_posse_mediano_acumulado,transferencias_ultimos_45d,indice_rotatividade_vrum
str,datetime[μs],u32,f64,u32,f64
"""CHASSI_SYN_000000001""",2026-03-28 14:23:58,0,0.0,0,0.0
"""CHASSI_SYN_000000002""",2026-03-03 14:24:56,0,0.0,0,0.0
"""CHASSI_SYN_000000002""",2026-03-07 10:24:04,1,0.0,1,1.0
"""CHASSI_SYN_000000002""",2026-03-24 04:37:35,2,3.832731,2,0.413845
"""CHASSI_SYN_000000003""",2026-02-04 05:36:07,0,0.0,0,0.0


**Resultado:** o índice exclui a proposta atual no numerador e na mediana. A primeira observação de cada chassi recebe histórico zero; isso será tratado como baixa confiança, não como evidência de legitimidade.

## O split temporal de 30 dias preserva ordem e maturidade do target?

Usamos três blocos consecutivos de 30 dias. A proposta sem target observado é retirada antes do treino e da avaliação.

In [9]:
FEATURES_NUM = ["valor_financiado", "valor_entrada", "prazo_meses", "idade_veiculo_anos", "ltv_fipe", "dias_desde_ultima_proposta", "qtd_propostas_historicas", "tempo_posse_mediano_acumulado", "transferencias_ultimos_7d", "transferencias_ultimos_15d", "transferencias_ultimos_30d", "transferencias_ultimos_45d", "transferencias_ultimos_90d", "indice_rotatividade_vrum", "flag_alternancia_pf_pj"]
FEATURES_CAT = ["tipo_proponente", "canal", "uf_proposta", "marca", "uf_registro"]
model_base = features.filter(pl.col("target_observado") == 1)
inicio = model_base.select(pl.col("data_hora_proposta").min()).item()
marco_30, marco_60 = inicio + timedelta(days=30), inicio + timedelta(days=60)
treino = model_base.filter(pl.col("data_hora_proposta") < marco_30)
validacao = model_base.filter((pl.col("data_hora_proposta") >= marco_30) & (pl.col("data_hora_proposta") < marco_60))
oot = model_base.filter(pl.col("data_hora_proposta") >= marco_60)
pl.DataFrame({"split": ["treino", "validacao", "oot"], "n": [treino.height, validacao.height, oot.height], "positivos": [int(treino["target_risco_90d"].sum()), int(validacao["target_risco_90d"].sum()), int(oot["target_risco_90d"].sum())], "inicio": [treino["data_hora_proposta"].min(), validacao["data_hora_proposta"].min(), oot["data_hora_proposta"].min()], "fim": [treino["data_hora_proposta"].max(), validacao["data_hora_proposta"].max(), oot["data_hora_proposta"].max()]})

split,n,positivos,inicio,fim
str,i64,i64,datetime[μs],datetime[μs]
"""treino""",725609,10659,2026-01-01 00:00:06,2026-01-30 23:59:48
"""validacao""",794643,11507,2026-01-31 00:00:07,2026-03-02 00:00:02
"""oot""",608961,9028,2026-03-02 00:00:06,2026-03-31 23:59:54


A validação define limiares e não recebe embaralhamento. O OOT simula propostas futuras; qualquer ganho precisa aparecer nele, não apenas no treino.

## O XGBoost generaliza para o bloco futuro?

Categorias são codificadas usando somente valores conhecidos no treino. Target, tipo de evento, data do risco, chassi e IF não entram como features.

In [10]:
def codificar(df, categorias=None):
    df = df.with_columns([pl.col(col).fill_null("DESCONHECIDO") for col in FEATURES_CAT])
    if categorias is None:
        categorias = {col: df.get_column(col).unique().sort().to_list() for col in FEATURES_CAT}
    nomes = []
    for col in FEATURES_CAT:
        for valor in categorias[col][1:]:
            nome = f"{col}_{valor}"
            df = df.with_columns((pl.col(col) == valor).cast(pl.Int8).alias(nome))
            nomes.append(nome)
    return df.drop(FEATURES_CAT), FEATURES_NUM + nomes, categorias

treino_x, feature_names, categorias = codificar(treino)
validacao_x, _, _ = codificar(validacao, categorias)
oot_x, _, _ = codificar(oot, categorias)
print("Features numéricas:", len(feature_names))

Features numéricas: 45


In [11]:
X_treino, X_validacao, X_oot = [df.select(feature_names).to_numpy() for df in [treino_x, validacao_x, oot_x]]
y_treino, y_validacao, y_oot = [df["target_risco_90d"].to_numpy().astype(np.int8) for df in [treino_x, validacao_x, oot_x]]
peso_positivo = float((y_treino == 0).sum() / (y_treino == 1).sum())
modelo = XGBClassifier(objective="binary:logistic", eval_metric="aucpr", n_estimators=200, max_depth=4, learning_rate=0.05, min_child_weight=10, subsample=0.8, colsample_bytree=0.8, reg_lambda=5.0, scale_pos_weight=peso_positivo, tree_method="hist", n_jobs=4, random_state=42)
modelo.fit(X_treino, y_treino, eval_set=[(X_validacao, y_validacao)], verbose=False)
scores = {"treino": modelo.predict_proba(X_treino)[:, 1], "validacao": modelo.predict_proba(X_validacao)[:, 1], "oot": modelo.predict_proba(X_oot)[:, 1]}

In [12]:
def ks(y, score):
    fpr, tpr, _ = roc_curve(y, score)
    return float(np.max(tpr - fpr))

def limiar_f1(y, score):
    precision, recall, thresholds = precision_recall_curve(y, score)
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    return float(thresholds[int(np.argmax(f1))])

limiar = limiar_f1(y_validacao, scores["validacao"])
def metricas(nome, y, score):
    pred = (score >= limiar).astype(np.int8)
    return {"split": nome, "n": len(y), "positivos": int(y.sum()), "taxa": float(y.mean()), "auc_roc": roc_auc_score(y, score), "ks": ks(y, score), "pr_auc": average_precision_score(y, score), "precision": precision_score(y, pred, zero_division=0), "recall": recall_score(y, pred, zero_division=0)}
metricas_modelo = pl.DataFrame([metricas("treino", y_treino, scores["treino"]), metricas("validacao", y_validacao, scores["validacao"]), metricas("oot", y_oot, scores["oot"])])
metricas_modelo

split,n,positivos,taxa,auc_roc,ks,pr_auc,precision,recall
str,i64,i64,f64,f64,f64,f64,f64,f64
"""treino""",725609,10659,0.01469,0.63916,0.187157,0.029589,0.014839,1.0
"""validacao""",794643,11507,0.014481,0.501388,0.007493,0.014414,0.014605,0.866168
"""oot""",608961,9028,0.014825,0.497786,0.002845,0.01474,0.014759,0.633695


**Resultado:** AUC e PR-AUC OOT são os gates principais. Treino acima de validação, com OOT próximo de baseline, significa sobreajuste ou ausência de sinal generalizável; não autoriza bloqueio.

## Como transformar score em política de três zonas?

Usamos cortes de capacidade definidos na validação: 5% para investigação e 1% para bloqueio simulado. Cortes não são probabilidades calibradas.

In [13]:
limiar_investigar = float(np.quantile(scores["validacao"], 0.95))
limiar_bloquear = float(np.quantile(scores["validacao"], 0.99))
def pontuar(df, score, nome):
    return df.with_columns(pl.Series("score_modelo", score), pl.lit(nome).alias("safra"))
pontuado = pl.concat([pontuar(treino, scores["treino"], "treino"), pontuar(validacao, scores["validacao"], "validacao"), pontuar(oot, scores["oot"], "oot")])
pontuado = pontuado.with_columns(pl.when(pl.col("score_modelo") < limiar_investigar).then(pl.lit("APROVAR")).when(pl.col("score_modelo") < limiar_bloquear).then(pl.lit("INVESTIGAR")).otherwise(pl.lit("BLOQUEAR")).alias("zona_decisao"))
pl.DataFrame({"regra": ["investigar", "bloquear"], "quantil_validacao": [0.95, 0.99], "limiar": [limiar_investigar, limiar_bloquear]})

regra,quantil_validacao,limiar
str,f64,f64
"""investigar""",0.95,0.541185
"""bloquear""",0.99,0.576746


In [14]:
def resumo_zona(df, safra):
    positivos_total = int(df["target_risco_90d"].sum())
    negativos_total = df.height - positivos_total
    return (df.group_by("zona_decisao").agg([pl.len().alias("n"), pl.col("target_risco_90d").sum().alias("positivos"), pl.col("valor_financiado").sum().alias("exposicao_total"), pl.when(pl.col("target_risco_90d") == 1).then(pl.col("valor_financiado")).otherwise(0).sum().alias("exposicao_risco")]).with_columns([pl.lit(safra).alias("safra"), (pl.col("positivos") / pl.col("n")).alias("taxa_risco_zona"), pl.when(pl.col("zona_decisao") == "APROVAR").then(0).otherwise(pl.col("positivos") / positivos_total).alias("captura_risco"), pl.when(pl.col("zona_decisao") == "APROVAR").then(0).otherwise((pl.col("n") - pl.col("positivos")) / negativos_total).alias("falso_positivo_global")]).sort("zona_decisao"))
politica = pl.concat([resumo_zona(pontuado.filter(pl.col("safra") == safra), safra) for safra in ["treino", "validacao", "oot"]])
politica.select(["safra", "zona_decisao", "n", "positivos", "taxa_risco_zona", "captura_risco", "falso_positivo_global", "exposicao_risco"])

safra,zona_decisao,n,positivos,taxa_risco_zona,captura_risco,falso_positivo_global,exposicao_risco
str,str,u32,i64,f64,f64,f64,f64
"""treino""","""APROVAR""",701628,9581,0.013655,0.0,0.0,1.0342e9
"""treino""","""BLOQUEAR""",2541,225,0.088548,0.021109,0.003239,1.9328e7
"""treino""","""INVESTIGAR""",21440,853,0.039785,0.080026,0.028795,8.4546e7
"""validacao""","""APROVAR""",754910,10963,0.014522,0.0,0.0,1.2018e9
"""validacao""","""BLOQUEAR""",7947,113,0.014219,0.00982,0.010003,8.0968e6
"""validacao""","""INVESTIGAR""",31786,431,0.013559,0.037455,0.040038,3.7553e7
"""oot""","""APROVAR""",590598,8769,0.014848,0.0,0.0,9.5287e8
"""oot""","""BLOQUEAR""",3898,57,0.014623,0.006314,0.006402,3.8490e6
"""oot""","""INVESTIGAR""",14465,202,0.013965,0.022375,0.023774,1.6518e7


`exposicao_risco` mostra valor financiado associado a casos rotulados como risco dentro da zona. Não é saving. Saving exige perda realizada, taxa de recuperação e custo de intervenção.

## A política é estável por safra e IF?

A IF é usada para monitoramento, não como sinal do modelo. Calculamos distribuição das zonas e ranking por IF no OOT.

In [15]:
estabilidade_safra = pontuado.group_by("safra").agg([pl.len().alias("n"), pl.col("target_risco_90d").mean().alias("taxa_risco"), (pl.col("zona_decisao") != "APROVAR").mean().alias("taxa_intervencao"), (pl.col("zona_decisao") == "BLOQUEAR").mean().alias("taxa_bloqueio")]).sort("safra")
estabilidade_safra

safra,n,taxa_risco,taxa_intervencao,taxa_bloqueio
str,u32,f64,f64,f64
"""oot""",608961,0.014825,0.030155,0.006401
"""treino""",725609,0.01469,0.033049,0.003502
"""validacao""",794643,0.014481,0.050001,0.010001


In [16]:
def resumo_if(grupo):
    y, score = grupo["target_risco_90d"].to_numpy(), grupo["score_modelo"].to_numpy()
    return {"safra": grupo["safra"][0], "if_id_sintetico": grupo["if_id_sintetico"][0], "n": grupo.height, "taxa_risco": float(y.mean()), "taxa_investigar": float((grupo["zona_decisao"] == "INVESTIGAR").mean()), "taxa_bloquear": float((grupo["zona_decisao"] == "BLOQUEAR").mean()), "auc_roc": float(roc_auc_score(y, score)) if np.unique(y).size == 2 else float("nan")}
estabilidade_if = pl.DataFrame([resumo_if(grupo) for grupo in pontuado.partition_by(["safra", "if_id_sintetico"], as_dict=True).values()])
estabilidade_if.group_by("safra").agg([pl.len().alias("ifs"), pl.col("taxa_investigar").min().alias("investigar_min"), pl.col("taxa_investigar").max().alias("investigar_max"), pl.col("taxa_bloquear").min().alias("bloquear_min"), pl.col("taxa_bloquear").max().alias("bloquear_max"), pl.col("auc_roc").min().alias("auc_min"), pl.col("auc_roc").max().alias("auc_max")]).sort("safra")

safra,ifs,investigar_min,investigar_max,bloquear_min,bloquear_max,auc_min,auc_max
str,u32,f64,f64,f64,f64,f64,f64
"""oot""",20,0.022203,0.024867,0.005616,0.00747,0.477484,0.513579
"""treino""",20,0.02809,0.030791,0.002872,0.004234,0.619084,0.666408
"""validacao""",20,0.037766,0.041234,0.00947,0.010988,0.482874,0.524781


**Resultado:** variação de taxa de encaminhamento entre safras ou IFs mede estabilidade operacional, não eficácia. AUC OOT próxima de 0,5 impede diferenciar IFs por risco do modelo.

## Qual exposição poderia ser evitada por intervenção?

Como não existe perda realizada, calculamos apenas cenário de saving proxy sobre `valor_financiado` em `INVESTIGAR` e `BLOQUEAR`. A zona `APROVAR` não recebe saving.

In [17]:
oot_politica = politica.filter(pl.col("safra") == "oot").with_columns([
    pl.when(pl.col("zona_decisao") == "APROVAR").then(0).otherwise(pl.col("exposicao_risco") * taxa).alias(f"saving_proxy_{int(taxa * 100)}pct")
    for taxa in [0.25, 0.50, 1.00]
])
oot_politica.select(["zona_decisao", "n", "positivos", "exposicao_risco", "saving_proxy_25pct", "saving_proxy_50pct", "saving_proxy_100pct"])

zona_decisao,n,positivos,exposicao_risco,saving_proxy_25pct,saving_proxy_50pct,saving_proxy_100pct
str,u32,i64,f64,f64,f64,f64
"""APROVAR""",590598,8769,9.5287e8,0.0,0.0,0.0
"""BLOQUEAR""",3898,57,3.8490e6,962239.6275,1.9245e6,3.8490e6
"""INVESTIGAR""",14465,202,1.6518e7,4.1295e6,8.2589e6,1.6518e7


**Resultado:** os valores são cenários financeiros condicionais. Decisão econômica exige medir perda evitada, custo de falso positivo e custo da mesa após implantação controlada.

## O que deve ser salvo para reprodução e governança?

Persistimos modelo, métricas, cortes e relatórios de política. Os valores financeiros são exposição ou saving proxy, nunca perda observada.

In [18]:
OUTPUT.mkdir(exist_ok=True)
modelo.save_model(OUTPUT / "processo_completo_xgboost_vrum.json")
metricas_modelo.write_csv(OUTPUT / "processo_completo_metricas_vrum.csv")
politica.write_csv(OUTPUT / "processo_completo_politica_vrum.csv")
estabilidade_if.write_csv(OUTPUT / "processo_completo_estabilidade_if_vrum.csv")
pl.DataFrame({"regra": ["investigar", "bloquear"], "quantil_validacao": [0.95, 0.99], "limiar_score": [limiar_investigar, limiar_bloquear]}).write_csv(OUTPUT / "processo_completo_limiares_vrum.csv")
print("Artefatos salvos em", OUTPUT)

Artefatos salvos em /var/home/andpax/Workspace/projetos/vscode-workspace/vrum/output


## Conclusão

- Target original tem `SEM_EVENTO` explícito, eventos de risco dentro da janela de 90 dias e uma proposta sem registro de evento. O caso sem registro não deve virar negativo automaticamente.
- Features temporais foram calculadas por chassi e com exclusão da proposta atual.
- Split temporal de 30 dias preserva simulação de produção.
- XGBoost só deve ser aceito se superar a prevalência no OOT.
- Política inicial: `APROVAR` provisório, `INVESTIGAR` em shadow mode ou mesa manual, `BLOQUEAR` desabilitado enquanto não houver evidência OOT e perdas reais.
- Próxima prioridade: validar regra de geração do target, obter perdas/recuperações reais e medir custo de falso positivo.